# LM（得分）检验统计量的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/simulation_lm_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：构造 Poisson 均值的 LM 统计量，模拟 H0（λ=10）与 H1（λ=11.5）下的分布。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页并固定种子。本 notebook 研究 Poisson 均值检验的 **LM（得分）检验统计量**。


In [ ]:
%%stata
clear
set more off
set seed 42


**test statistic**

定义统计量程序：`lm_test_stat` 生成 n 个 Poisson(λ) 样本，返回 `chi2_stat=(Σ(x-10))²/(n·10)`。
**推导思路**：Poisson 模型关于 λ 的得分函数为 Σ(x_i-λ)/λ、信息量为 n/λ；在 λ0=10 处取值的 LM 统计量 = [Σ(x_i-λ0)]²/(n·λ0)，即程序公式。LM 检验的优点是**只需要在原假设下估计**（解释变量只需在 λ0 处取值），在本例中直接给出显式公式。原假设下渐近 χ²(1)。


In [ ]:
%%stata
cap program drop lm_test_stat
program define lm_test_stat, rclass
	syntax [,obs(integer 100) lambda(real 10)]
	clear
	tempvar x x_10
	quietly{
		set obs `obs'
		gen `x'=rpoisson(`lambda')
		gen `x_10'=`x'-10
		su `x_10'
		return scalar chi2_stat=(r(sum))^2/(`obs'*10)
	}
end


**upper tail of chi2(1)**

计算临界值 χ²_{0.95}(1)。


In [ ]:
%%stata
local cv=invchi2tail(1,0.05)


**simulation under H0**

**H0 下的模拟**：默认 λ=10（原假设成立）重复5000次，统计量直方图与 χ²(1) 密度、临界值对比；`su p_005` 给出实际拒绝率。


In [ ]:
%%stata
simulate chi2=r(chi2_stat),reps(5000): lm_test_stat
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005


**simulation under H1**

**H1 下的模拟**：以 λ=11.5（备择假设）重复5000次，统计量分布右移、拒绝率上升，展示检验功效。


In [ ]:
%%stata
clear
simulate chi2=r(chi2_stat),reps(5000): lm_test_stat, lambda(11.5)
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005
